# Exploratory data analysis - Federal Reserve interest rates

This notebook is the narrative companion to `scripts/run_eda.py`. It reads the
validated monthly panel produced by the ingestion pipeline and reproduces the
statistics quoted in the README and `docs/dataset.md`. Nothing here is
hard-coded: run the pipeline first, then execute the cells.

```
python scripts/run_pipeline.py --stage ingest
python scripts/run_eda.py
```


In [ ]:
from pathlib import Path

import pandas as pd

from interest_rate_mlops.config import load_data_config
from interest_rate_mlops.eda import autocorrelation, build_eda_report, summarise_panel

data_cfg = load_data_config()
panel = pd.read_parquet(data_cfg.processed_dir + "/panel.parquet")
print(panel.shape)
print(panel.index.min(), "->", panel.index.max())
panel.head()

## Series summary

Observation counts, dispersion and persistence. The lag-1 autocorrelation near
1.0 is why lag and rolling features dominate: the series is close to a random
walk with drift, and a naive carry-forward baseline is genuinely strong.


In [ ]:
summary = summarise_panel(panel)
summary

In [ ]:
target = data_cfg.target.slug
print(f"autocorrelation lag 1 : {autocorrelation(panel[target], 1):.4f}")
print(f"autocorrelation lag 12: {autocorrelation(panel[target], 12):.4f}")
print(f"peak {panel[target].max():.2f}% on {panel[target].idxmax().date()}")
print(
    f"trough {panel[target][panel[target] > 0].min():.2f}% on {panel[target][panel[target] > 0].idxmin().date()}"
)

## Figures and the generated report

Writes the same artifacts as the CLI (`reports/eda/`): history, distribution,
rolling statistics, autocorrelation and seasonality plots plus a markdown report.


In [ ]:
payload = build_eda_report(
    panel,
    Path("reports/eda"),
    target=data_cfg.target.slug,
    unit=data_cfg.target.unit,
)
payload["window"]

## What the analysis changed in the design

1. **Frequency** - the raw file has 904 rows but only 753 monthly observations:
   151 intra-month policy-decision rows carry a target rate and no effective
   rate. They are dropped, never interpolated.
2. **Lags in observations, not days** - `lag_12` means twelve months, and the
   panel is anchored to month start.
3. **Regime shift** - rates run from 0.07% (2011) to 19.10% (1981). A random
   chronological split leaves the test block in a low-rate regime, which is why
   fold-to-fold CV dispersion is reported next to the test error.
4. **MAPE is secondary** - near-zero rates break percentage errors, so MAE and
   RMSE decide, and MAPE is reported with its coverage share.
